In [ ]:
import pandas as pd

file_path = r"C:\Users\HP\Downloads\Messy_Employee_dataset.csv"  

with open(file_path, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f.readline())

Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work

EMP1000,Bob,Davis,25,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,TRUE

EMP1001,Bob,Brown,,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,TRUE



In [5]:
df_raw = pd.read_csv(
    file_path,
    sep=",",
    encoding="utf-8"
)

print(df_raw.shape)
df_raw.head()

(1020, 12)


,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 88.8 KB


In [7]:
df_raw.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [ ]:
dtype_map = {
    "Employee_ID": "str",
    "First_Name": "str",
    "Last_Name": "str",
    "Age": "Int64",           
    "Department_Region": "str",
    "Status": "str",
    "Join_Date": "str",       
    "Salary": "float64",
    "Email": "str",
    "Phone": "str",           
    "Performance_Score": "str",
}

df_clean = pd.read_csv(
    file_path,
    sep=",",
    encoding="utf-8",
    dtype=dtype_map
)

df_clean.dtypes

Employee_ID              str
First_Name               str
Last_Name                str
Age                    Int64
Department_Region        str
Status                   str
Join_Date                str
Salary               float64
Email                    str
Phone                    str
Performance_Score        str
Remote_Work             bool
dtype: object

In [9]:
df_clean["Join_Date"] = pd.to_datetime(
    df_clean["Join_Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

df_clean["Join_Date"]

0      2021-04-02
1      2020-07-10
2      2023-12-07
3      2021-11-27
4      2022-01-05
          ...    
1015   2023-08-19
1016   2021-11-07
1017   2023-10-04
1018   2024-12-16
1019   2021-02-22
Name: Join_Date, Length: 1020, dtype: datetime64[us]

In [10]:
df_clean["Join_Date"].isnull().sum()

np.int64(0)

In [11]:
categorical_cols = ["Status", "Performance_Score", "Department_Region"]

for col in categorical_cols:
    df_clean[col] = df_clean[col].str.strip().str.title()

df_clean[categorical_cols].head()

,Status,Performance_Score,Department_Region
0,Active,Average,Devops-California
1,Active,Excellent,Finance-Texas
2,Pending,Good,Admin-Nevada
3,Inactive,Good,Admin-Nevada
4,Active,Poor,Cloud Tech-Florida


In [ ]:
df_clean["Department_Region"] = df_raw["Department_Region"].str.strip()

df_clean[["Status", "Performance_Score", "Department_Region"]].head()

,Status,Performance_Score,Department_Region
0,Active,Average,DevOps-California
1,Active,Excellent,Finance-Texas
2,Pending,Good,Admin-Nevada
3,Inactive,Good,Admin-Nevada
4,Active,Poor,Cloud Tech-Florida


In [ ]:
phone_negativos = (df_clean["Phone"].astype(str).str.startswith("-")).sum()
print("Teléfonos negativos:", phone_negativos)

df_clean["Phone"].head(10)

Teléfonos negativos: 1020


0    -1651623197
1    -1898471390
2    -5596363211
3    -3476490784
4    -1586734256
5    -5409003485
6    -4518376063
7    -4134327559
8    -4177656123
9    -8156985699
Name: Phone, dtype: str

## Auditoría: columna `Phone`

Se detectó que el 100% de los valores de `Phone` (1020 de 1020) son negativos y no
tienen formato de número telefónico válido (ej: `-1651623197`). Dado que el dataset
es sintético y no hay forma de recuperar un valor real ni confirmar que "corregirlo"
(ej: quitando el signo negativo) produzca un dato verdadero, se decide **no modificar**
el valor original. En su lugar, se marca la columna como inválida mediante una
columna auxiliar `Phone_Invalido`, dejando el dato crudo disponible para quien
necesite investigar la causa raíz con el equipo de origen de los datos.

In [14]:
df_clean["Phone_Invalido"] = df_clean["Phone"].astype(str).str.startswith("-")
df_clean[["Phone", "Phone_Invalido"]].head()

,Phone,Phone_Invalido
0,-1651623197,True
1,-1898471390,True
2,-5596363211,True
3,-3476490784,True
4,-1586734256,True


In [15]:
df_clean["Age"].describe()

count        809.0
mean     32.484549
std        5.65686
min           25.0
25%           25.0
50%           30.0
75%           40.0
max           40.0
Name: Age, dtype: Float64

In [16]:
df_clean["Age"].value_counts().sort_index()

Age
25    206
30    205
35    188
40    210
Name: count, dtype: Int64

## Auditoría: nulos en `Age` y `Salary`

- `Age`: 211 de 1020 filas (~20.7%) sin valor. La distribución de los valores
  existentes es discreta (solo 25, 30, 35, 40) y bastante pareja entre sí
  (~200 filas cada una). Imputar con la mediana u otra técnica proporcional
  duplicaría artificialmente una categoría, distorsionando el análisis y
  fabricando un dato que no existe. Se decide **dejar los nulos sin imputar**
  y documentarlos, para que cualquier análisis posterior sobre `Age` los
  excluya explícitamente (ej: `.dropna()` o reportando el % de datos faltantes).

- `Salary`: 24 de 1020 filas (~2.4%) sin valor. Se mantiene el mismo criterio
  por consistencia: no se imputa, se documenta como dato faltante.

In [17]:
age_nulos = df_clean["Age"].isnull().sum()
salary_nulos = df_clean["Salary"].isnull().sum()

print(f"Age: {age_nulos} nulos ({age_nulos/len(df_clean)*100:.1f}%)")
print(f"Salary: {salary_nulos} nulos ({salary_nulos/len(df_clean)*100:.1f}%)")

Age: 211 nulos (20.7%)
Salary: 24 nulos (2.4%)


In [ ]:
print("Filas duplicadas:", df_clean.duplicated().sum())

print("Employee_ID duplicados:", df_clean["Employee_ID"].duplicated().sum())

Filas duplicadas: 0
Employee_ID duplicados: 0


In [19]:
print("=== COMPARACIÓN DE TIPOS ===")
comparacion = pd.DataFrame({
    "dtype_original": df_raw.dtypes.astype(str),
    "dtype_limpio": df_clean.dtypes.astype(str)
})
comparacion

=== COMPARACIÓN DE TIPOS ===


,dtype_original,dtype_limpio
Age,float64,Int64
Department_Region,str,str
Email,str,str
Employee_ID,str,str
First_Name,str,str
Join_Date,str,datetime64[us]
Last_Name,str,str
Performance_Score,str,str
Phone,int64,str
Phone_Invalido,NaN,bool


In [20]:
print("=== NULOS FINALES EN df_clean ===")
df_clean.isnull().sum()

=== NULOS FINALES EN df_clean ===


Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
Phone_Invalido         0
dtype: int64

In [21]:
df_clean.head(10)

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Phone_Invalido
0,EMP1000,Bob,Davis,25,DevOps-California,Active,2021-04-02,59767.65,bob.davis@example.com,-1651623197,Average,True,True
1,EMP1001,Bob,Brown,<NA>,Finance-Texas,Active,2020-07-10,65304.66,bob.brown@example.com,-1898471390,Excellent,True,True
2,EMP1002,Alice,Jones,<NA>,Admin-Nevada,Pending,2023-12-07,88145.90,alice.jones@example.com,-5596363211,Good,True,True
3,EMP1003,Eva,Davis,25,Admin-Nevada,Inactive,2021-11-27,69450.99,eva.davis@example.com,-3476490784,Good,True,True
4,EMP1004,Frank,Williams,25,Cloud Tech-Florida,Active,2022-01-05,109324.61,frank.williams@example.com,-1586734256,Poor,False,True
5,EMP1005,Alice,Garcia,40,Sales-Texas,Inactive,2020-06-10,88642.84,alice.garcia@example.com,-5409003485,Good,False,True
6,EMP1006,Frank,Jones,<NA>,Admin-Nevada,Active,2020-04-03,96288.43,frank.jones@example.com,-4518376063,Good,False,True
7,EMP1007,Bob,Jones,30,Cloud Tech-Florida,Inactive,2022-07-17,94497.91,bob.jones@example.com,-4134327559,Average,True,True
8,EMP1008,Frank,Davis,35,Admin-Nevada,Inactive,2023-12-08,115565.82,frank.davis@example.com,-4177656123,Excellent,True,True
9,EMP1009,Charlie,Johnson,<NA>,DevOps-New York,Active,2022-08-04,76561.88,charlie.johnson@example.com,-8156985699,Excellent,True,True


In [22]:
output_path = "employees_clean.csv"

df_clean.to_csv(output_path, index=False)

print(f"Archivo guardado en: {output_path}")

Archivo guardado en: employees_clean.csv


In [23]:
df_verificacion = pd.read_csv(output_path)

print("=== TIPOS AL RELEER EL CSV EXPORTADO ===")
print(df_verificacion.dtypes)
print()
print("=== NULOS AL RELEER EL CSV EXPORTADO ===")
print(df_verificacion.isnull().sum())

=== TIPOS AL RELEER EL CSV EXPORTADO ===
Employee_ID              str
First_Name               str
Last_Name                str
Age                  float64
Department_Region        str
Status                   str
Join_Date                str
Salary               float64
Email                    str
Phone                  int64
Performance_Score        str
Remote_Work             bool
Phone_Invalido          bool
dtype: object

=== NULOS AL RELEER EL CSV EXPORTADO ===
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
Phone_Invalido         0
dtype: int64


In [24]:
df_final_check = pd.read_csv(output_path, dtype=dtype_map)
df_final_check["Join_Date"] = pd.to_datetime(df_final_check["Join_Date"], errors="coerce")

print(df_final_check.dtypes)

Employee_ID                     str
First_Name                      str
Last_Name                       str
Age                           Int64
Department_Region               str
Status                          str
Join_Date            datetime64[us]
Salary                      float64
Email                           str
Phone                           str
Performance_Score               str
Remote_Work                    bool
Phone_Invalido                 bool
dtype: object


## Resumen del proceso de limpieza

Se aplicó un flujo de 5 etapas (Load → Inspect → Clean → Review → Export) sobre un
dataset sintético de 1020 empleados con problemas de tipos, fechas y datos inválidos.

**Decisiones clave:**
- `Age` se convirtió a `Int64` (entero que admite nulos) para corregir un silent
  type casting que pandas aplicaba automáticamente por la presencia de valores faltantes.
- `Join_Date` se convirtió a `datetime64` con `pd.to_datetime(errors='coerce')`;
  no se encontraron fechas inválidas (0 valores `NaT`).
- `Phone` (100% de los valores negativos e inválidos) no se "corrigió" artificialmente:
  se documentó como dato sintético inválido y se marcó con una columna auxiliar
  (`Phone_Invalido`), evitando fabricar datos sin evidencia real.
- Los nulos de `Age` (20.7%) y `Salary` (2.4%) no se imputaron: la distribución
  discreta y pareja de `Age` habría distorsionado el dataset con una imputación
  simple. Se dejaron como nulos y documentados.
- Al exportar a CSV se confirmó que los `dtype` definidos (`Int64`, `datetime64`,
  `str` para `Phone`) no persisten en el archivo — deben volver a aplicarse
  explícitamente (`dtype_map`) al recargar el CSV limpio.

**Resultado:** `employees_clean.csv`, con tipos correctos, categorías estandarizadas,
duplicados verificados (0) y datos inválidos documentados en vez de ocultados.